# MPI Communication Modes Demo (mpi4py)

This notebook demonstrates the 8 core MPI point-to-point communication operations:

| Operation | Description |
|---|---|
| `MPI_BSEND` | Append outgoing message to a local send buffer |
| `MPI_SEND` | Send a message and wait until copied to local or remote buffer |
| `MPI_SSEND` | Send a message and wait until transmission starts |
| `MPI_SENDRECV` | Send a message and wait for reply |
| `MPI_ISEND` | Pass reference to outgoing message, and continue |
| `MPI_ISSEND` | Pass reference to outgoing message, and wait until receipt starts |
| `MPI_RECV` | Receive a message; block if there is none |
| `MPI_IRECV` | Check if there is an incoming message, but do not block |

**How this notebook works:** each example is a tiny, self-contained script. We write it to disk with
the `%%writefile` magic, then launch it with the MPI process launcher so you can see two MPI ranks
talking to each other. Every example uses exactly 2 processes: **rank 0** sends, **rank 1** receives.

## Setup

**Windows (your case):**
1. Install Microsoft MPI — download and run **both** installers from
   https://www.microsoft.com/en-us/download/details.aspx?id=105289 :
   - `MSMpiSetup.exe` (the runtime — this is what provides `mpiexec`)
   - `msmpisdk.msi` (the SDK — needed to build mpi4py)
2. Close and reopen your terminal/Anaconda Prompt so PATH updates take effect, then confirm with:
   `mpiexec -help`
3. Install mpi4py. Conda is more reliable than pip on Windows:
   ```
   conda install -c conda-forge mpi4py
   ```
   (or `pip install mpi4py` if you're not using conda)




## Function Reference

### `MPI.COMM_WORLD`
The default **communicator** — the group of all processes launched by `mpirun`/`mpiexec` for this
program. Almost every MPI call is a method on a communicator object.
- **Type:** `MPI.Intracomm` object (not a function call — it's a ready-made instance)

### `comm.Get_rank()`
Returns this process's unique ID within the communicator.
- **Parameters:** none
- **Returns:** `int` — the calling process's rank, from `0` to `comm.Get_size() - 1`





In [1]:
# Detect the right MPI launcher for this machine and confirm mpi4py is installed
import shutil, sys

if shutil.which("mpiexec"):
    MPI_LAUNCHER = "mpiexec"          # Windows (MS-MPI) or some Linux MPI distros
elif shutil.which("mpirun"):
    MPI_LAUNCHER = "mpirun --oversubscribe"   # OpenMPI on Linux/Mac
else:
    raise RuntimeError(
        "No MPI launcher (mpiexec/mpirun) found on PATH. "
        "See the setup instructions above, then restart the kernel."
    )

PYTHON_EXE = sys.executable   # use the exact same Python this notebook is running under

print("Using MPI launcher:", MPI_LAUNCHER)
print("Using Python:", PYTHON_EXE)

import mpi4py
print("mpi4py version:", mpi4py.__version__)


Using MPI launcher: mpiexec
Using Python: D:\PhD-Stuff\Development\EEGSeizurePrediction\env-eegSP\Scripts\python.exe
mpi4py version: 4.1.2


## 1. `MPI_SEND` / `MPI_RECV` — the basic blocking pair

- `MPI_SEND`: the call **returns once the message has been copied** into a local or remote buffer —
  it does not necessarily mean the receiver has picked it up yet.
- `MPI_RECV`: **blocks** the calling process until a matching message arrives.

### `comm.send(obj, dest, tag=0)`
Blocking standard send (`MPI_SEND`). Serializes `obj` with pickle and sends it to another rank.
- **`obj`** — any picklable Python object (dict, list, string, number, NumPy array, etc.) to send
- **`dest`** (`int`) — the rank of the process that should receive the message
- **`tag`** (`int`, optional, default `0`) — a numeric label used to match this send with a specific
  `recv` call, useful when a rank might get several different kinds of messages
- **Returns:** `None`. The call returns once the message has been copied into a local or remote
  buffer — not necessarily once the receiver has picked it up

### `comm.recv(source, tag=0)`
Blocking receive (`MPI_RECV`). Waits until a matching message arrives, then returns it.
- **`source`** (`int`) — the rank to receive from (use `MPI.ANY_SOURCE` to accept from any rank)
- **`tag`** (`int`, optional, default `0`) — must match the `tag` used by the sender (use
  `MPI.ANY_TAG` to accept any tag)
- **Returns:** the unpickled Python object that was sent (same type as what `send` was given)

In [2]:
%%writefile send_recv.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           

if rank == 0:                                    # branch: code only rank 0 runs
    data = {"msg": "hello", "value": 42}         # the Python object we will send (any picklable object works)
    comm.send(data, dest=1, tag=0)               # MPI_SEND: blocking send; returns None once data is copied to a buffer
    print(f"[Rank {rank}] Sent: {data}")         # confirm what rank 0 sent
elif rank == 1:                                  # branch: code only rank 1 runs
    data = comm.recv(source=0, tag=0)            # MPI_RECV: blocks until a matching message arrives; returns the sent object
    print(f"[Rank {rank}] Received: {data}")     # confirm what rank 1 received
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")                # safety net so unexpected ranks don't fail silently


Writing send_recv.py


In [3]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} send_recv.py

[Rank 0] Sent: {'msg': 'hello', 'value': 42}
[Rank 1] Received: {'msg': 'hello', 'value': 42}


## 2. `MPI_BSEND` — buffered send

`MPI_BSEND` copies the outgoing message into a **user-supplied buffer** that you attach ahead of
time with `MPI.Attach_buffer`. The call returns as soon as the copy is done, *even if the receiver
isn't ready yet* — the buffer holds the message until MPI can deliver it. This trades memory for
guaranteed non-blocking-ish behavior without needing a request object.

### `MPI.Attach_buffer(buf)` / `MPI.Detach_buffer()`
Attach/detach the memory region that `comm.bsend` will copy outgoing messages into.
- **`buf`** — a writable memory buffer (e.g. from `MPI.Alloc_mem`)
- **Returns:** `None` for `Attach_buffer`; `Detach_buffer` returns the buffer object that was
  attached, so it can be freed

### `MPI.Alloc_mem(size)` / `MPI.Free_mem(buf)`
Allocate/free a raw block of memory suitable for use as an MPI buffer.
- **`size`** (`int`) — number of bytes to allocate
- **`buf`** — the buffer object previously returned by `Alloc_mem`
- **Returns:** `Alloc_mem` returns a writable buffer object; `Free_mem` returns `None`

### `comm.bsend(obj, dest, tag=0)`
Buffered send (`MPI_BSEND`). Same parameters and return type as `comm.send`, but copies `obj` into
a buffer you attached with `MPI.Attach_buffer` first, so it can return without waiting on the
receiver at all.
- **Returns:** `None`

In [2]:
%%writefile bsend.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           

# Every rank that will call bsend must first attach a buffer big enough
# to hold the message.
BUFFER_SIZE = 10_000                             # size in bytes of the buffer bsend will copy messages into
buf = MPI.Alloc_mem(BUFFER_SIZE)                 # MPI.Alloc_mem(size) -> buffer object: allocates BUFFER_SIZE bytes of raw memory
MPI.Attach_buffer(buf)                           # MPI.Attach_buffer(buf) -> None: registers buf as the buffer future bsend calls will use

if rank == 0:                                    # branch: code only rank 0 runs
    data = {"msg": "buffered hello"}             # the Python object we will send
    comm.bsend(data, dest=1, tag=0)              # MPI_BSEND: copies data into the attached buffer and returns immediately (None)
    print(f"[Rank {rank}] bsend returned immediately, message is sitting in the buffer")  # confirm bsend did not block
elif rank == 1:                                  # branch: code only rank 1 runs
    data = comm.recv(source=0, tag=0)            # MPI_RECV: blocks until the buffered message arrives; returns the sent object
    print(f"[Rank {rank}] Received: {data}")     # confirm what rank 1 received
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")    
    

MPI.Detach_buffer()                              # MPI.Detach_buffer() -> buffer object: unregisters the buffer (waits for pending bsends to finish)
MPI.Free_mem(buf)                                # MPI.Free_mem(buf) -> None: releases the raw memory back to the system


Overwriting bsend.py


In [3]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} bsend.py

[Rank 0] bsend returned immediately, message is sitting in the buffer
[Rank 1] Received: {'msg': 'buffered hello'}


## 3. `MPI_SSEND` — synchronous send

`MPI_SSEND` only returns once the **receiver has started receiving** the message — it performs a
real handshake between sender and receiver. This is useful when you need to be sure the other side
is actually listening before your code moves on (e.g. for correctness or timing experiments).

In the example below, rank 1 sleeps for a second before calling `recv`, so you can see rank 0's
`ssend` visibly wait for it.

### `comm.ssend(obj, dest, tag=0)`
Synchronous send (`MPI_SSEND`). Same parameters and return type as `comm.send`, but only returns
once the receiving rank has **started** its matching `recv`.
- **Returns:** `None`

In [6]:
%%writefile ssend.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
import time                                      # used to simulate rank 1 being busy before it receives
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           # comm.Get_rank() -> int: this process's unique ID (0, 1, ...)

if rank == 0:                                    # branch: code only rank 0 runs
    data = "synchronous hello"                   # the Python object we will send
    print(f"[Rank {rank}] About to ssend...")    # marker printed before the (potentially slow) ssend call
    comm.ssend(data, dest=1, tag=0)              # MPI_SSEND: blocks until rank 1 has started its matching recv; returns None
    print(f"[Rank {rank}] ssend completed: receiver has started receiving")  # only prints after the handshake completes
elif rank == 1:                                  # branch: code only rank 1 runs
    time.sleep(1)                                # simulate the receiver being busy first, so ssend above visibly waits
    data = comm.recv(source=0, tag=0)            # MPI_RECV: blocks until a matching message arrives; returns the sent object
    print(f"[Rank {rank}] Received: {data}")     # confirm what rank 1 received
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")    


Writing ssend.py


In [7]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} ssend.py

[Rank 1] Received: synchronous hello
[Rank 0] About to ssend...
[Rank 0] ssend completed: receiver has started receiving


## 4. `MPI_SENDRECV` — send and receive in one call

`MPI_SENDRECV` combines a send and a receive into a **single call**. This is the standard tool for
exchanging data between two ranks (or around a ring of ranks) without risking deadlock — you don't
have to worry about ordering `send`/`recv` calls correctly on both sides.


### `comm.sendrecv(sendobj, dest, sendtag=0, source, recvtag=0)`
Combined send + receive (`MPI_SENDRECV`) in one deadlock-safe call.
- **`sendobj`** — the picklable object to send
- **`dest`** (`int`) — rank to send `sendobj` to
- **`sendtag`** (`int`, optional, default `0`) — tag attached to the outgoing message
- **`source`** (`int`) — rank to receive the reply from
- **`recvtag`** (`int`, optional, default `0`) — tag the incoming message must match
- **Returns:** the object received from `source` (the send half returns nothing separately)

In [8]:
%%writefile sendrecv.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           # comm.Get_rank() -> int: this process's unique ID (0, 1, ...)

partner = 1 - rank                               # compute the other rank's ID: rank 0 <-> rank 1

sent_data = f"hello from {rank}"                 # the Python object this rank will send to its partner
received_data = comm.sendrecv(sent_data, dest=partner, sendtag=0,   # MPI_SENDRECV: sends sent_data to `partner`
                               source=partner, recvtag=0)            # ...and simultaneously receives a reply from `partner`; returns the received object

print(f"[Rank {rank}] Sent: '{sent_data}'  |  Received: '{received_data}'")  # show both halves of the exchange


Writing sendrecv.py


In [9]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} sendrecv.py

[Rank 0] Sent: 'hello from 0'  |  Received: 'hello from 1'
[Rank 1] Sent: 'hello from 1'  |  Received: 'hello from 0'


## 5. `MPI_ISEND` — non-blocking send

`MPI_ISEND` hands MPI a *reference* to the outgoing message and returns **immediately** with a
`request` object, letting the sender keep doing other work. You later call `request.wait()` (or
`request.test()`) to confirm the send has actually completed. Use this to overlap communication
with computation.

### `request.wait()`
Blocks until the non-blocking operation (`isend`/`issend`/`irecv`) that produced this `request` has
completed.
- **Parameters:** none
- **Returns:** for a request from `irecv`, returns the received object; for `isend`/`issend`,
  returns `None`

### `comm.isend(obj, dest, tag=0)`
Non-blocking send (`MPI_ISEND`). Same `obj`/`dest`/`tag` meaning as `comm.send`, but returns
immediately instead of waiting for the copy to finish.
- **Returns:** an `MPI.Request` object — a handle you use to check on / wait for completion

In [10]:
%%writefile isend.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           # comm.Get_rank() -> int: this process's unique ID (0, 1, ...)

if rank == 0:                                    # branch: code only rank 0 runs
    data = "non-blocking hello"                  # the Python object we will send
    req = comm.isend(data, dest=1, tag=0)        # MPI_ISEND: starts a non-blocking send; returns an MPI.Request handle immediately
    print(f"[Rank {rank}] isend issued, doing other work while it completes...")  # proves isend did not block
    # ... you could do useful computation here ...
    req.wait()                                   # request.wait() -> None: blocks until the isend has actually completed
    print(f"[Rank {rank}] isend confirmed complete")  # only prints after the send is truly done
elif rank == 1:                                  # branch: code only rank 1 runs
    data = comm.recv(source=0, tag=0)            # MPI_RECV: blocks until a matching message arrives; returns the sent object
    print(f"[Rank {rank}] Received: {data}")     # confirm what rank 1 received
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")    


Writing isend.py


In [11]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} isend.py

[Rank 0] isend issued, doing other work while it completes...
[Rank 0] isend confirmed complete
[Rank 1] Received: non-blocking hello


## 6. `MPI_ISSEND` — non-blocking synchronous send

`MPI_ISSEND` is the non-blocking version of `MPI_SSEND`: it returns a request immediately, but that
request only completes (i.e. `wait()` only returns) once the **receiver has started receiving**.
This combines not blocking right away with still guaranteeing a handshake happened.


### `comm.issend(obj, dest, tag=0)`
Non-blocking synchronous send (`MPI_ISSEND`). Same parameters as `comm.send`.
- **Returns:** an `MPI.Request` object whose `.wait()` only completes once the receiver has started
  receiving


In [12]:
%%writefile issend.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
import time                                      # used to simulate rank 1 being busy before it receives
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           # comm.Get_rank() -> int: this process's unique ID (0, 1, ...)

if rank == 0:                                    # branch: code only rank 0 runs
    data = "non-blocking synchronous hello"      # the Python object we will send
    req = comm.issend(data, dest=1, tag=0)       # MPI_ISSEND: starts a non-blocking synchronous send; returns an MPI.Request handle immediately
    print(f"[Rank {rank}] issend issued, request not yet complete")  # proves issend did not block at call time
    req.wait()                                   # request.wait() -> None: blocks until rank 1 has started receiving
    print(f"[Rank {rank}] issend confirmed: receiver has started receiving")  # only prints after the handshake completes
elif rank == 1:                                  # branch: code only rank 1 runs
    time.sleep(1)                                # simulate the receiver being busy first, so issend's wait() visibly blocks
    data = comm.recv(source=0, tag=0)            # MPI_RECV: blocks until a matching message arrives; returns the sent object
    print(f"[Rank {rank}] Received: {data}")     # confirm what rank 1 received
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")


Writing issend.py


In [13]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} issend.py

[Rank 1] Received: non-blocking synchronous hello
[Rank 0] issend issued, request not yet complete
[Rank 0] issend confirmed: receiver has started receiving


## 7. `MPI_RECV` — blocking receive (recap)

We already used `MPI_RECV` above, but to call it out explicitly: `MPI_RECV` **blocks** the calling
process until a matching message shows up. If no message is coming, the process will wait forever
— so every `recv` needs a matching `send`/`bsend`/`ssend`/`isend`/`issend` somewhere.

In [14]:
%%writefile recv_only.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           # comm.Get_rank() -> int: this process's unique ID (0, 1, ...)

if rank == 0:                                    # branch: code only rank 0 runs
    comm.send("a message that recv is waiting for", dest=1, tag=0)  # MPI_SEND: blocking send; returns None once copied to a buffer
elif rank == 1:                                  # branch: code only rank 1 runs
    print(f"[Rank {rank}] Blocking here until a message arrives...")  # printed before the blocking call
    data = comm.recv(source=0, tag=0)            # MPI_RECV: blocks the process until a matching message shows up; returns the sent object
    print(f"[Rank {rank}] Unblocked! Received: {data}")  # only prints once recv has unblocked
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")    
    


Writing recv_only.py


In [15]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} recv_only.py

[Rank 1] Blocking here until a message arrives...
[Rank 1] Unblocked! Received: a message that recv is waiting for


## 8. `MPI_IRECV` — non-blocking receive

`MPI_IRECV` posts a receive **without blocking** — it immediately returns a `request` object. The
process can then do other work, and later call `request.wait()` to actually get the data (or
`request.test()` to just check "has it arrived yet?" without blocking at all).

In the example below, rank 0 deliberately delays sending, so you can see rank 1 stay busy instead
of blocking while it waits.


### `comm.irecv(source, tag=0)`
Non-blocking receive (`MPI_IRECV`). Same `source`/`tag` meaning as `comm.recv`, but posts the
receive and returns immediately instead of blocking.
- **Returns:** an `MPI.Request` object — call `.wait()` on it later to actually get the data

In [16]:
%%writefile irecv.py
from mpi4py import MPI                          # import the mpi4py bindings to MPI
import time                                      # used to simulate rank 0 delaying its send
comm = MPI.COMM_WORLD                            # get the default communicator (all launched processes)
rank = comm.Get_rank()                           # comm.Get_rank() -> int: this process's unique ID (0, 1, ...)

if rank == 0:                                    # branch: code only rank 0 runs
    time.sleep(1)                                # simulate rank 0 being slow to send, so rank 1's irecv has time to "do other work"
    comm.send("delayed hello", dest=1, tag=0)    # MPI_SEND: blocking send; returns None once copied to a buffer
elif rank == 1:                                  # branch: code only rank 1 runs
    req = comm.irecv(source=0, tag=0)            # MPI_IRECV: posts a non-blocking receive; returns an MPI.Request handle immediately (no message yet)
    print(f"[Rank {rank}] irecv posted, doing other work while waiting...")  # proves irecv did not block at call time
    # ... you could do useful computation here ...
    data = req.wait()                            # request.wait() -> object: blocks until the message arrives, then returns it
    print(f"[Rank {rank}] Received: {data}")     # confirm what rank 1 received
else:                                             # branch: any rank other than 0 or 1 
    print(f"Unknown Rank {rank}")    


Writing irecv.py


In [17]:
!{MPI_LAUNCHER} -n 2 {PYTHON_EXE} irecv.py

[Rank 1] irecv posted, doing other work while waiting...
[Rank 1] Received: delayed hello


## Summary

| Operation | Blocking? | Waits for... |
|---|---|---|
| `MPI_BSEND` | No (returns after local copy) | Nothing — buffered locally |
| `MPI_SEND` | Yes | Message copied to a buffer (local or remote) |
| `MPI_SSEND` | Yes | Receiver to **start** receiving |
| `MPI_SENDRECV` | Yes | Both the send and the matching receive |
| `MPI_ISEND` | No | Nothing at call time — check with `request.wait()`/`test()` |
| `MPI_ISSEND` | No | Nothing at call time, but `request.wait()` blocks until receiver starts |
| `MPI_RECV` | Yes | A matching message to arrive |
| `MPI_IRECV` | No | Nothing at call time — check with `request.wait()`/`test()` |

**Rule of thumb:** use blocking calls (`SEND`/`RECV`/`SENDRECV`) when correctness and simplicity
matter more than overlap; use non-blocking calls (`ISEND`/`IRECV`/`ISSEND`) when you want to
overlap communication with computation; use `BSEND` when you need a send to return immediately and
are willing to manage your own buffer.